# 02 — Phishing Dataset Overview

**Purpose (§4 step 2, PHISHING_MODEL_GUIDE):** Inspect the phishing dataset before training.

Checklist:
- [ ] Confirm correct HF subset/config name → update `HF_PHISHING_EMAIL_SUBSET` in `loaders.py` if needed
- [ ] Check column names and dtypes
- [ ] Check label distribution (phishing vs legit balance)
- [ ] Sample rows — is `text` plain body or RFC 2822 with headers?
- [ ] Language mix — English vs Indonesian
- [ ] Inspect Enron CSV parsing (legit class)
- [ ] Confirm `_parse_raw_email()` output looks correct
- [ ] Check `build_phishing_input()` final output format

> **Do not import this notebook from `src/`. Exploration only (CLAUDE.md §6).**
> Strip output before committing: `nbstripout notebooks/02_phishing_overview.ipynb`

In [ ]:
import sys, os
from pathlib import Path

# Find repo root: walk up until we find CLAUDE.md (works regardless of
# where VS Code / Jupyter starts the kernel)
_cwd = Path().absolute()
for _p in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_p / 'CLAUDE.md').exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError(f"Can't find repo root from {_cwd}. Open VS Code at prior-mail-model/.")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import warnings
warnings.filterwarnings('ignore')

print('Repo root :', REPO_ROOT)
print('Working dir:', os.getcwd())
print('Python     :', sys.version)

## 1. Discover available HF dataset configs

We hardcoded `HF_PHISHING_EMAIL_SUBSET = "emails"` in `loaders.py` — confirm it exists.

In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names('ealvaradob/phishing-dataset', trust_remote_code=True)
print('Available configs:', configs)

# ✅ ACTION: if 'emails' is NOT in this list, find the correct email config
# and update HF_PHISHING_EMAIL_SUBSET in src/data/loaders.py

## 2. Load the email subset

In [ ]:
from datasets import load_dataset

# Confirmed available configs: ['texts', 'urls', 'webs', 'combined_full', 'combined_reduced']
# 'texts' contains email + SMS data — correct choice for email phishing model
HF_SUBSET = 'texts'

raw = load_dataset('ealvaradob/phishing-dataset', name=HF_SUBSET, trust_remote_code=True)
print('Splits:', raw)
print()
for split in raw:
    print(f'{split}: {raw[split].num_rows} rows, columns={raw[split].column_names}')

In [ ]:
import pandas as pd

# Pool all splits (ealvaradob ships only 'train')
from datasets import concatenate_datasets
hf_all = concatenate_datasets([raw[s] for s in raw])

df_hf = hf_all.to_pandas()
print('Shape:', df_hf.shape)
print()
print('Dtypes:')
print(df_hf.dtypes)
print()
print('Null counts:')
print(df_hf.isnull().sum())

## 3. Label distribution

In [ ]:
print('Label distribution:')
print(df_hf['label'].value_counts())
print()
print('Class balance:')
vc = df_hf['label'].value_counts(normalize=True)
print(f'  0 (legit):   {vc.get(0, 0):.1%}')
print(f'  1 (phishing): {vc.get(1, 0):.1%}')

# ✅ Note the imbalance — will inform phishing_class_multiplier in config

## 4. Inspect `text` column — plain body or RFC 2822 email?

This determines whether `_parse_raw_email()` will extract headers or fall back to plain body.

In [ ]:
# Sample a few phishing and legit examples
phishing_samples = df_hf[df_hf['label'] == 1]['text'].head(3).tolist()
legit_samples    = df_hf[df_hf['label'] == 0]['text'].head(3).tolist()

print('=== PHISHING SAMPLES ===')
for i, t in enumerate(phishing_samples):
    print(f'--- sample {i} (first 300 chars) ---')
    print(t[:300])
    print()

print('=== LEGIT SAMPLES ===')
for i, t in enumerate(legit_samples):
    print(f'--- sample {i} (first 300 chars) ---')
    print(t[:300])
    print()

In [ ]:
# Test _parse_raw_email on the HF samples
import sys; sys.path.insert(0, '.')
from src.data.loaders import _parse_raw_email

print('=== _parse_raw_email on phishing examples ===')
for i, t in enumerate(phishing_samples):
    sender, subject, body = _parse_raw_email(t)
    print(f'[{i}] sender={sender!r}')
    print(f'     subject={subject!r}')
    print(f'     body[:100]={body[:100]!r}')
    print()

# ✅ ACTION: if sender+subject are always empty, the text is plain body only
# (still works — build_phishing_input handles empty sender/subject)
# But note: sender signal is lost, which weakens phishing detection!

In [ ]:
# What fraction of rows have detectable RFC 2822 headers?
import email as _em

def has_headers(text):
    try:
        msg = _em.message_from_string(text)
        return bool(msg.get('From') or msg.get('Subject'))
    except Exception:
        return False

sample = df_hf.sample(min(500, len(df_hf)), random_state=42)
frac_with_headers = sample['text'].apply(has_headers).mean()
print(f'Fraction with RFC 2822 headers: {frac_with_headers:.1%}')
print('(If < 10%, text is plain body — sender signal not available in this dataset)')

## 5. Language mix

The dataset is expected to be English-heavy — confirming this justifies the mBERT base model choice (§11 decision).

In [ ]:
from langdetect import detect, LangDetectException

def safe_detect(text):
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return 'unknown'

sample = df_hf.sample(min(300, len(df_hf)), random_state=42)
langs = sample['text'].apply(safe_detect)

print('Language distribution (sample of 300):')
print(langs.value_counts().head(10))
print()
print(f'English:    {(langs=="en").mean():.1%}')
print(f'Indonesian: {(langs=="id").mean():.1%}')

# ✅ Expected: mostly English → mBERT is the right choice

## 6. Enron CSV — legit class inspection

In [ ]:
import pandas as pd

df_enron = pd.read_csv('emails.csv', nrows=10)
print('Columns:', df_enron.columns.tolist())
print('Shape (first 10):', df_enron.shape)
print()
print('Sample message (first 400 chars):')
print(df_enron['message'].iloc[0][:400])

In [ ]:
# Verify _parse_raw_email on Enron emails
from src.data.loaders import _parse_raw_email

print('=== _parse_raw_email on Enron examples ===')
for i in range(3):
    sender, subject, body = _parse_raw_email(df_enron['message'].iloc[i])
    print(f'[{i}] sender={sender!r}')
    print(f'     subject={subject!r}')
    print(f'     body[:80]={body[:80]!r}')
    print()
# ✅ Expected: Enron has proper RFC 2822 headers → sender always parsed

## 7. Final model input format — `build_phishing_input()`

In [ ]:
from src.data.preprocess import build_phishing_input

# Example from HF phishing dataset
sender_hf, subject_hf, body_hf = _parse_raw_email(phishing_samples[0])
model_input_hf = build_phishing_input(sender_hf, subject_hf, body_hf)
print('HF phishing model_input (first 200 chars):')
print(model_input_hf[:200])
print()

# Example from Enron legit dataset  
sender_en, subject_en, body_en = _parse_raw_email(df_enron['message'].iloc[0])
model_input_en = build_phishing_input(sender_en, subject_en, body_en)
print('Enron legit model_input (first 200 chars):')
print(model_input_en[:200])

## 8. Full loader smoke test

In [ ]:
from src.data.loaders import load_phishing_dataset

ds = load_phishing_dataset(
    enron_csv_path='emails.csv',
    legit_sample_size=1000,  # small sample for notebook speed
    seed=42,
)

train_ds = ds['train']
print('Train split:', train_ds)
print()
print('Columns:', train_ds.column_names)
print()

import pandas as pd
df = train_ds.to_pandas()
print('Label distribution:')
print(df['phishing'].value_counts())
print()
print('Sample rows:')
df[['sender_email', 'subject', 'phishing']].head(5)

## 9. Summary & action items

Fill in after running the notebook:

| Check | Result | Action |
|---|---|---|
| HF subset name | `"emails"` ✅ / ❌ `___` | Update `HF_PHISHING_EMAIL_SUBSET` in loaders.py if wrong |
| HF rows | ___ total, ___ phishing, ___ legit | — |
| Text has RFC 2822 headers | ___% | If < 10%, sender signal unavailable — note in model card |
| Language mix | ___% EN, ___% ID | Should be mostly EN → mBERT confirmed |
| Enron parsing | sender ✅ / ❌ | — |
| model_input format | Looks correct ✅ / ❌ | — |
| Label imbalance | ___:___ ratio | Adjust `phishing_class_multiplier` in config if far from 1:1 |

Once all checks pass → `make data-phishing` then Colab training (§5).